# Trash-ICRA19 Exploration (PRIMARY DATASET)
**Assigned to:** ___
**No GPU needed. CPU runtime is fine.**

This is our MAIN training dataset. Everything depends on understanding this correctly.

In [ ]:
import os, time
from google.colab import drive
drive.mount('/content/drive')

DATASET = "/content/drive/MyDrive/underwater_datasets/trash_ICRA19"

# Auto-detect if path is slightly different
if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for name in os.listdir(base):
        if "icra" in name.lower() or "trash_icra" in name.lower():
            DATASET = os.path.join(base, name)
            break

assert os.path.exists(DATASET), f"NOT FOUND. Available: {os.listdir(base)}"
print(f"✓ Dataset: {DATASET}")


In [ ]:
# === FULL FOLDER STRUCTURE ===
from pathlib import Path
from collections import Counter

root = Path(DATASET)
for dirpath, dirnames, filenames in sorted(os.walk(root)):
    depth = str(dirpath).replace(str(root), "").count(os.sep)
    if depth > 3: continue
    indent = "  " * depth
    exts = Counter(Path(f).suffix.lower() for f in filenames)
    print(f"{indent}{os.path.basename(dirpath)}/")
    for ext, cnt in sorted(exts.items()):
        print(f"{indent}  {cnt:>6} {ext}")


In [ ]:
# === PER-SPLIT STATS ===
from pathlib import Path

root = Path(DATASET)
for split in ["train", "val", "test"]:
    d = root / split
    if not d.exists():
        print(f"  ✗ {split}/ NOT FOUND")
        continue
    jpgs = set(f.stem for f in d.glob("*.jpg"))
    txts = set(f.stem for f in d.glob("*.txt"))
    print(f"\n--- {split.upper()} ---")
    print(f"  Images: {len(jpgs)}")
    print(f"  Labels: {len(txts)}")
    print(f"  Images without labels: {len(jpgs - txts)}")
    print(f"  Labels without images: {len(txts - jpgs)}")
    if jpgs - txts:
        print(f"    Examples: {list(jpgs - txts)[:3]}")


In [ ]:
# === CLASS DISTRIBUTION (CRITICAL) ===
from collections import Counter, defaultdict
from pathlib import Path

root = Path(DATASET)
overall = Counter()
per_split = defaultdict(Counter)
bad = 0

for split in ["train", "val", "test"]:
    d = root / split
    if not d.exists(): continue
    for txt in d.glob("*.txt"):
        try:
            for line in open(txt):
                parts = line.strip().split()
                if len(parts) == 5:
                    cls = int(parts[0])
                    overall[cls] += 1
                    per_split[split][cls] += 1
        except: bad += 1

total = sum(overall.values())
print("="*65)
print(f"{'Class':>6} {'train':>8} {'val':>8} {'test':>8} {'Total':>8} {'%':>7}")
print("-"*65)
for cls in sorted(overall):
    t = per_split["train"].get(cls,0)
    v = per_split["val"].get(cls,0)
    te = per_split["test"].get(cls,0)
    pct = overall[cls]/total*100
    print(f"  {cls:>4} {t:>8} {v:>8} {te:>8} {overall[cls]:>8} {pct:>6.1f}%")
print("-"*65)
print(f"Total: {total} boxes | Classes: {len(overall)} | Errors: {bad}")

mx, mn = max(overall.values()), min(overall.values())
print(f"\nImbalance ratio: {mx/mn:.1f}x" if mn > 0 else "")


In [ ]:
# === BOUNDING BOX VALIDATION ===
from pathlib import Path
root = Path(DATASET)

issues = {"outside_01": 0, "zero_dim": 0, "too_large": 0, "too_small": 0}
total = 0

for txt in root.rglob("*.txt"):
    try:
        for line in open(txt):
            parts = line.strip().split()
            if len(parts) != 5: continue
            try:
                c, x, y, w, h = int(parts[0]), *map(float, parts[1:])
                total += 1
                if not (0<=x<=1 and 0<=y<=1): issues["outside_01"] += 1
                if w<=0 or h<=0: issues["zero_dim"] += 1
                if w>0.95 or h>0.95: issues["too_large"] += 1
                if w*h < 0.0001: issues["too_small"] += 1
            except ValueError: pass
    except: pass

print(f"Total boxes checked: {total}")
for name, cnt in issues.items():
    s = "✓" if cnt == 0 else "⚠️"
    print(f"  {s} {name}: {cnt} ({cnt/total*100:.2f}%)" if total else f"  {name}: {cnt}")


In [ ]:
# === IMAGE SIZE ANALYSIS ===
import cv2, numpy as np
from pathlib import Path
from collections import Counter

root = Path(DATASET)
all_imgs = sorted(root.rglob("*.jpg"))
step = max(1, len(all_imgs)//200)
samples = all_imgs[::step]

widths, heights = [], []
sizes = Counter()

print(f"Sampling {len(samples)} of {len(all_imgs)} images...")
for p in samples:
    img = cv2.imread(str(p))
    if img is None: continue
    h, w = img.shape[:2]
    widths.append(w); heights.append(h)
    sizes[f"{w}x{h}"] += 1

print(f"\nWidth:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
print(f"Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")
print(f"\nMost common sizes:")
for s, c in sizes.most_common(5):
    print(f"  {s}: {c}")


In [ ]:
# === BBOX SIZE DISTRIBUTION ===
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path

root = Path(DATASET)
areas, bw_list, bh_list, per_img = [], [], [], []

for txt in root.rglob("*.txt"):
    count = 0
    try:
        for line in open(txt):
            p = line.strip().split()
            if len(p)==5:
                w,h = float(p[3]), float(p[4])
                if 0<w<=1 and 0<h<=1:
                    bw_list.append(w); bh_list.append(h)
                    areas.append(w*h); count+=1
        if count: per_img.append(count)
    except: pass

fig, ax = plt.subplots(2,2, figsize=(14,10))
ax[0,0].hist(bw_list, 50, color="#2196F3", edgecolor="white"); ax[0,0].set_title("Box Widths")
ax[0,1].hist(bh_list, 50, color="#4CAF50", edgecolor="white"); ax[0,1].set_title("Box Heights")
ax[1,0].hist(areas, 50, color="#FF9800", edgecolor="white"); ax[1,0].set_title("Box Areas (w×h)")
ax[1,1].hist(per_img, bins=range(0,max(per_img)+2), color="#9C27B0", edgecolor="white"); ax[1,1].set_title("Objects Per Image")
plt.suptitle("Trash-ICRA19 — BBox Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/trash_icra19_bbox.png", dpi=150, bbox_inches="tight"); plt.show()

small = sum(1 for a in areas if a<0.01)
med = sum(1 for a in areas if 0.01<=a<0.1)
big = sum(1 for a in areas if a>=0.1)
print(f"Small (<1%): {small} ({small/len(areas)*100:.1f}%)")
print(f"Medium (1-10%): {med} ({med/len(areas)*100:.1f}%)")
print(f"Large (>10%): {big} ({big/len(areas)*100:.1f}%)")
print(f"Objects/image: mean={np.mean(per_img):.1f}, max={max(per_img)}")


In [ ]:
# === SAMPLE IMAGES WITH BBOXES ===
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

root = Path(DATASET)
train = root / "train"
if not train.exists():
    train = next(d for d in root.iterdir() if d.is_dir() and list(d.glob("*.jpg")))

jpgs = sorted(train.glob("*.jpg"))
np.random.seed(42)
picks = [jpgs[i] for i in np.random.choice(len(jpgs), min(9,len(jpgs)), replace=False)]
COLORS = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),(0,255,255),(128,0,255),(255,128,0),(0,128,255),(128,255,0)]

fig, axes = plt.subplots(3,3, figsize=(18,18))
for idx, p in enumerate(picks):
    ax = axes[idx//3][idx%3]
    img = cv2.imread(str(p))
    if img is None: ax.axis("off"); continue
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    txt = p.with_suffix(".txt")
    nb = 0
    if txt.exists():
        for line in open(txt):
            parts = line.strip().split()
            if len(parts)==5:
                try:
                    c,cx,cy,bw,bh = int(parts[0]),*map(float,parts[1:])
                    x1,y1 = int((cx-bw/2)*W), int((cy-bh/2)*H)
                    x2,y2 = int((cx+bw/2)*W), int((cy+bh/2)*H)
                    cv2.rectangle(rgb,(x1,y1),(x2,y2),COLORS[c%len(COLORS)],2)
                    cv2.putText(rgb,str(c),(x1,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.6,COLORS[c%len(COLORS)],2)
                    nb+=1
                except: pass
    ax.imshow(rgb); ax.set_title(f"{p.name} ({nb} boxes)", fontsize=9); ax.axis("off")

plt.suptitle("Trash-ICRA19 — Samples with BBoxes", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig("/content/trash_icra19_samples.png", dpi=150, bbox_inches="tight"); plt.show()


In [ ]:
# === FILL THIS SUMMARY AND SHARE WITH TEAM ===
print('''
================================================================
TRASH-ICRA19 EXPLORATION SUMMARY
================================================================
1. Train / Val / Test images: ___ / ___ / ___
2. Number of classes: ___
3. Class mapping:
   0: ___
   1: ___
   (fill all)
4. ROV class present? ___
5. Imbalance ratio: ___x
6. Bad annotations: ___
7. Most common image size: ___
8. Small objects (<1% area): ___%
9. Issues found: ___

READY FOR TRAINING: YES / NO
================================================================
''')
